# 第71课 · 下一个词怎么选？——贪婪解码与集束搜索（beam search），让 Whisper 开口说话

**学习目标**
- 理解自回归（autoregressive，AR）解码：每步用之前的 token 预测下一个
- 实现贪婪解码（greedy）和简单 beam search（width=2）
- 了解 Whisper 的特殊 token 体系

← **上一课**　[L70 · Whisper 架构解析](L70_whisper_arch.ipynb)

> 上节课学习了 **Whisper 架构解析**：Log-Mel 输入、Transformer Encoder-Decoder、多任务头。  
> 本课将探讨 **Whisper 解码策略**。

## 本课剧情：Siri 说出第一个字——然后呢？

语音识别的最后一步，是把编码器输出的向量转换成文字。Whisper 不是一次性输出整句话——它是**自回归**的，像人类说话一样，一个词一个词地"想"出来。

**问题**：每一步都有完整的词汇表（Whisper 有 ~51,000 个 BPE token）。怎么从概率分布里选下一个词？

**两种策略**：

| 策略 | 做法 | 类比 |
|---|---|---|
| 贪婪解码 (Greedy) | 每步选概率最大的token | 棋手每步选局部最优走法 |
| Beam Search (宽度W) | 同时追踪W条"候选句子"，每步扩展所有候选 | 棋手同时推演W条路线，保留累计得分最高的 |

**为什么贪婪不够好**？

```
步骤1: P("我")=0.6, P("你")=0.4   → 贪婪选"我"
步骤2: P("我爱")=0.3, P("我恨")=0.7 → 贪婪路径变成"我恨..."
步骤2: P("你好")=0.9               → "你好"路径更好！
```

贪婪在第1步的"局部最优"，可能让第2步走向"全局最差"。Beam Search 同时保留"我"和"你"两条路线，最终选择总对数概率更高的那条。

**本节任务**：用玩具 LM（固定概率分布）分别实现贪婪解码和 Beam Search，数值验证两者的差异。

In [ ]:
import numpy as np

In [ ]:
# 模拟一个玩具 LM：词汇表大小 10，5 步解码
np.random.seed(7)
VOCAB_SIZE = 10
EOT = 9            # end-of-text token id
MAX_STEPS = 8

def fake_lm(context: list) -> np.ndarray:
    """玩具语言模型：基于上文输出 logits。（模拟，真实中是 Transformer 前向传播）"""
    rng = np.random.RandomState(sum(context) % 137)
    logits = rng.randn(VOCAB_SIZE)
    if len(context) >= 4:
        logits[EOT] += 2.0   # 超过 4 步后更容易结束
    return logits

def softmax(logits):
    x = logits - logits.max()
    e = np.exp(x)
    return e / e.sum()

## ✏️ 任务 1：贪婪解码

**签名**：
```python
def greedy_decode(prompt: list, max_steps: int = MAX_STEPS, eot: int = EOT) -> list:
    # 返回: list[int] — 包含 prompt 和生成 token 的完整序列（含 EOT）
```

**3步实现**：

| 步骤 | 操作 | 细节 |
|---|---|---|
| 1 | 初始化序列为 `prompt` 副本 | 不修改输入 |
| 2 | 循环 max_steps 次：调用 `fake_lm(seq)` 取 logits，`argmax` 得 next_token | logits shape = (VOCAB_SIZE,) |
| 3 | 若 next_token == eot 则追加 EOT 后 break；否则 append 继续 | 到达 max_steps 不提前终止 |

**验收标准**：
- 输出为 list[int]，长度 ∈ [len(prompt)+1, len(prompt)+max_steps+1]
- 末尾含 EOT token（或达到 max_steps）
- seed=7 的玩具 LM 下结果确定（不随机）

In [ ]:
def greedy_decode(prompt: list, max_steps: int = MAX_STEPS, eot: int = EOT) -> list:
    """贪婪解码：每步选概率最大的 token。

    算法步骤提示：
      1. 初始化 tokens = list(prompt)
      2. 循环 max_steps 次：
         a. logits = fake_lm(tokens)
         b. probs  = softmax(logits)
         c. next_token = int(np.argmax(probs))   # 取概率最大的 token
         d. tokens.append(next_token)
         e. 若 next_token == eot：break
      3. 返回 tokens
    """
    # ✏️ TODO: 实现此函数
    raise NotImplementedError("TODO")

prompt = [0, 1, 2]   # 模拟 [<|startoftranscript|>, <|en|>, <|transcribe|>]
try:
    result = greedy_decode(prompt)
    print(f'贪婪解码结果: {result}')
    # 验证 1：前缀保持不变
    assert result[0:3] == prompt, '前缀应保持不变'
    # 验证 2：以 EOT 结束或达到最大步数（终止条件）
    assert result[-1] == EOT or len(result) - len(prompt) == MAX_STEPS, (
        f'应以 EOT 结束或达到 MAX_STEPS={MAX_STEPS} 步，实际序列: {result}'
    )
    print('✅ 贪婪解码验证通过')
except (NotImplementedError, TypeError):
    result = None
    print('⚠️  greedy_decode 尚未实现，result 设为 None（占位）')


## ✏️ 任务 2：Beam Search（宽度 W=2）

**签名**：
```python
def beam_search(prompt: list, width: int = 2, max_steps: int = MAX_STEPS, eot: int = EOT) -> list:
    # 返回: list[int] — 累计对数概率最高的完整序列
```

**Beam 数据结构**：每条候选为 `(log_prob, sequence)`：
```python
beams = [(0.0, list(prompt))]   # 初始化：1条beam，对数概率=0
```

**4步算法**：

| 步骤 | 操作 | 关键点 |
|---|---|---|
| 1 | 对每条 beam：调 `fake_lm(seq)` → softmax → log → 取 top-W token | width×width 候选 |
| 2 | 所有候选按 `log_prob + log(token_prob)` 排序，取 top-W | 累计对数概率 |
| 3 | 若某 beam 末尾 == EOT：移入 completed | 允许不同 beam 在不同步终止 |
| 4 | 若 beams 空或全完成：返回 completed 中得分最高的 | 取第一个元素（最高分） |

**验收标准**：
- seed=7，W=2：beam search 结果与贪婪结果**可能不同**（验证两条路线的差异）
- 返回值是对数概率最高的序列（不是所有候选）

In [ ]:
def beam_search(
    prompt: list, width: int = 2, max_steps: int = MAX_STEPS, eot: int = EOT
) -> list:
    """简单 beam search。返回分数最高的序列。

    分数定义：该路径所有 log P(token_i) 之和（越大越好）。
      score(seq) = Σ log P(seq[i] | seq[:i])  for i in range(len(prompt), len(seq))

    算法步骤提示：
      1. 初始化 beams = [(0.0, list(prompt))]   # (累积 log-prob, token 序列)
      2. 循环 max_steps 次：
         a. candidates = []
         b. 对每个 (score, tokens) in beams：
              logits = fake_lm(tokens)
              probs  = softmax(logits)
              for token_id in range(VOCAB_SIZE):
                  new_score = score + np.log(probs[token_id] + 1e-12)
                  candidates.append((new_score, tokens + [token_id]))
         c. 按 score 降序排列 candidates，取前 width 个
         d. 命中 EOT 的 beam 移入 completed，不再扩展；若无存活 beam 则 break
      3. 返回 completed + beams 中分数最高的序列
    """
    # ✏️ TODO: 实现此函数
    raise NotImplementedError("TODO")

try:
    beam_result = beam_search(prompt, width=2)
    print(f'Beam search 结果 (W=2): {beam_result}')
    if result is not None:
        print(f'贪婪结果:               {result}')
    print('（两者可能不同——beam search 探索更多路径）')

    # 验证 1：返回值以 prompt 为前缀
    assert beam_result[:len(prompt)] == prompt, 'beam_result 应以 prompt 为前缀'

    # 验证 2：beam search log-prob ≥ greedy log-prob（最优性保证）
    if result is not None:
        def compute_logprob(seq):
            score = 0.0
            tokens = list(prompt)
            for tok in seq[len(prompt):]:
                logits = fake_lm(tokens)
                probs = softmax(logits)
                score += np.log(probs[tok] + 1e-12)
                tokens.append(tok)
            return score
        beam_score   = compute_logprob(beam_result)
        greedy_score = compute_logprob(result)
        assert beam_score >= greedy_score - 1e-6, (
            f'beam search 分数 ({beam_score:.4f}) 应 ≥ greedy 分数 ({greedy_score:.4f})'
        )
    print('✅ Beam search 验证通过')
except (NotImplementedError, TypeError):
    print('⚠️  beam_search 尚未实现，跳过验证')


## 🔗 Aurora 连接

本课实现的两个解码算法与 Aurora 代码库的对应关系：

| 本课实现 | Aurora 模块 | 说明 |
|---|---|---|
| `greedy_decode` | `aurora.llm.greedy_decode`（`src/aurora/llm/sample.py`） | src 版是单步 argmax（对一帧 logits 选最大）；本课把它扩展成完整的自回归解码循环 |
| `beam_search` | 本课 notebook 实现 | src 中尚未收录 beam search；Whisper 论文长音频转写设定用 beam_size=5 |

`aurora.llm` 目前提供单步的 `greedy_decode` 与 `top_k_sample` / `top_p_sample`；
本课的贪婪循环与 beam search 演示如何把"单步选 token"组装成完整序列解码。
L86 将在此基础上系统学习 temperature / top-k / top-p 采样（`aurora.llm.top_k_sample`）。


## 本课收束

| 策略 | 复杂度 | 特点 |
|---|---|---|
| 贪婪 | O(T×V) | 快，局部最优 |
| Beam W=2 | O(T×W×V) | 更好，代价是 W 倍计算 |
| Top-k / Top-p | O(T×V) | 引入随机性，避免重复（见 L86）|

Whisper 温度 0 时默认贪婪解码；论文的长音频转写设定用 beam_size=5 + length_penalty 换取更高质量。

下一步 L72：在真实数据集上用 LoRA 微调 Whisper。

## ✏️ 白板挑战：解码策略手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：贪婪解码的时间复杂度是多少？设序列长度 T，词表大小 V。  
（每步 argmax 是 O(V)，共 T 步，总计 O(T×V)）

**问 2**：Beam Search（宽度 W）每步扩展后有多少候选序列？取 top-W 后剩几条？  
（扩展后 W×V 条候选；取 top-W 后保留 W 条）

**问 3**：若 W=1，Beam Search 退化成什么？  
（退化为贪婪解码——每步只保留1条beam，等价于argmax）

**问 4**：为什么 Beam Search 用对数概率相加，而不是概率相乘？  
（避免浮点下溢：0.1^100 ≈ 0，但 100×log(0.1) = -230，数值安全；加法也更快）

**问 5**：Whisper 推理时默认用哪种解码策略？为什么？  
（openai-whisper 在温度 0 时默认贪婪解码；论文的长音频转写设定用 beam_size=5。贪婪更快，beam search 更准，可按需在速度和质量间权衡）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：贪婪复杂度演示（O(T×V)步骤计数）
try:
    import time
    np.random.seed(7)
    prompt_test = [0]
    t0 = time.perf_counter()
    result_greedy = greedy_decode(prompt_test, max_steps=MAX_STEPS)
    t1 = time.perf_counter()
    assert isinstance(result_greedy, list), "greedy_decode应返回list"
    assert result_greedy[0] == 0, "序列应以prompt开头"
    # 终止条件：以 EOT 结束，或到达 MAX_STEPS（本玩具 LM 下 [0] 起步不一定命中 EOT）
    assert result_greedy[-1] == EOT or len(result_greedy) - len(prompt_test) == MAX_STEPS, (
        f"序列应以 EOT 结束或达到 MAX_STEPS={MAX_STEPS} 步，实际: {result_greedy}"
    )
    print(f"Q1 ✅  greedy_decode: {len(result_greedy)}个token，每步O(V={VOCAB_SIZE})，总O({len(result_greedy)*VOCAB_SIZE})")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  贪婪复杂度=O(T×V)，T=序列长度，V={VOCAB_SIZE}（待实现后验证）")

# 问2：beam search候选数
try:
    np.random.seed(7)
    result_beam = beam_search([0], width=2, max_steps=MAX_STEPS)
    assert isinstance(result_beam, list), "beam_search应返回list"
    assert result_beam[0] == 0, "序列应以prompt开头"
    W = 2
    print(f"Q2 ✅  Beam W={W}: 每步扩展后{W}×{VOCAB_SIZE}={W*VOCAB_SIZE}候选，取top-{W}保留{W}条")
    # 检查是否与贪婪不同（演示beam的价值）
    if result_greedy != result_beam:
        print(f"       ✨ beam结果{result_beam} ≠ 贪婪结果{result_greedy}（beam找到了更好路径！）")
    else:
        print(f"       beam结果与贪婪相同（玩具LM下有时相同，真实LM通常不同）")
except (NotImplementedError, TypeError):
    print(f"Q2 ✅  Beam W=2: 每步W×V=2×{VOCAB_SIZE}={2*VOCAB_SIZE}候选，取top-2（待实现后验证）")

# 问3：W=1退化
try:
    np.random.seed(7)
    result_w1 = beam_search([0], width=1, max_steps=MAX_STEPS)
    # W=1时beam应与greedy结果相同
    assert result_w1 == result_greedy, f"W=1时beam={result_w1}应等于greedy={result_greedy}"
    print(f"Q3 ✅  W=1 beam search = greedy decode: {result_w1}")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  W=1退化为贪婪（只保留1条beam = argmax）（待实现后验证）")
except AssertionError as e:
    print(f"Q3 ⚠️  {e}（检查beam_search(width=1)是否正确实现）")

# 问4：对数概率（数值验证）
probs_q4 = np.array([0.1] * 10)   # 简单均匀分布
log_probs = np.log(probs_q4 + 1e-10)
assert np.all(np.isfinite(log_probs)), "log_probs应全有限"
# 100步概率乘积 vs 对数概率相加
prob_product = np.prod(probs_q4[:5])
log_sum = np.sum(log_probs[:5])
print(f"Q4 ✅  概率乘积={prob_product:.2e}（接近下溢），对数和={log_sum:.3f}（数值安全）")

# 问5：Whisper默认策略（概念验证）
print(f"Q5 ✅  Whisper温度0时默认贪婪解码；论文长音频设定用beam_size=5（可用--beam_size调节）；贪婪=最快，beam=最准")
print("\n🎉 Whisper解码策略白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l71_review = {
    "greedy_vs_beam_tradeoff":  None,  # 理解贪婪局部最优vs beam全局搜索的权衡？True/False
    "log_prob_rationale":       None,  # 理解为什么用对数概率相加（避免下溢）？True/False
    "greedy_decode_impl":       None,  # greedy_decode()实现正确，含EOT，seed=7结果确定？True/False
    "beam_search_impl":         None,  # beam_search(width=2)实现正确，返回最高累计对数概率序列？True/False
    "whiteboard_passed":        None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l71_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l71_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L71 全部通关！进入 L72：Whisper 微调')

---

→ **下一课**　[L72 · Whisper 微调](L72_whisper_finetune.ipynb)

> 下节课将学习 **Whisper 微调**：LoRA 低秩注入 vs 全参数，中文 / 方言数据适配实战。